In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv(r"C:\Users\ASUS\Downloads\DAML dataset\tech_advertising_campaigns_dataset.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (10000, 41)


,campaign_id,campaign_objective,platform,ad_placement,device_type,operating_system,creative_format,creative_size,ad_copy_length,has_call_to_action,...,avg_session_duration_seconds,pages_per_session,industry_vertical,budget_tier,CTR,CPC,conversion_rate,CPA,ROAS,profit
0,CAMP_00001,Lead Generation,Facebook,Search,Mobile,Android,Text,728x90,Short,False,...,70,2.00,E-commerce,High,0.606,2.23,0.498,448.23,0.32,-606.79
1,CAMP_00002,Engagement,Facebook,Feed,Mobile,iOS,Image,320x50,Long,True,...,105,2.74,Finance,Medium,0.932,2.48,1.071,231.63,1.49,570.04
2,CAMP_00003,Conversions,Google Ads,Feed,Tablet,iOS,Video,1920x1080,Short,False,...,149,3.97,Healthcare,Low,1.033,3.88,5.357,72.43,6.68,2468.70
3,CAMP_00004,Conversions,LinkedIn,Search,Desktop,iOS,Carousel,1920x1080,Short,False,...,147,2.49,Education,Medium,1.573,8.29,2.439,339.89,0.49,-517.39
4,CAMP_00005,Brand Awareness,Facebook,Stories,Mobile,iOS,Image,1920x1080,Short,False,...,81,3.04,SaaS,Low,1.409,1.41,0.000,0.00,0.00,-425.82


In [3]:
missing_values = df.isnull().sum()

print("Total missing values:", missing_values.sum())

missing_values[missing_values > 0]

Total missing values: 0


Series([], dtype: int64)

In [4]:
duplicate_count = df.duplicated().sum()

print("Total duplicate rows:", duplicate_count)

Total duplicate rows: 0


In [5]:
numerical_features = df.select_dtypes(include=np.number).columns.tolist()
categorical_features = df.select_dtypes(include="object").columns.tolist()
boolean_features = df.select_dtypes(include="bool").columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

print("\nBoolean Features:")
print(boolean_features)

Numerical Features:
['creative_age_days', 'quarter', 'hour_of_day', 'campaign_day', 'quality_score', 'actual_cpc', 'impressions', 'clicks', 'conversions', 'ad_spend', 'revenue', 'bounce_rate', 'avg_session_duration_seconds', 'pages_per_session', 'CTR', 'CPC', 'conversion_rate', 'CPA', 'ROAS', 'profit']

Categorical Features:
['campaign_id', 'campaign_objective', 'platform', 'ad_placement', 'device_type', 'operating_system', 'creative_format', 'creative_size', 'ad_copy_length', 'creative_emotion', 'target_audience_age', 'target_audience_gender', 'audience_interest_category', 'income_bracket', 'purchase_intent_score', 'start_date', 'day_of_week', 'industry_vertical', 'budget_tier']

Boolean Features:
['has_call_to_action', 'retargeting_flag']


In [6]:
df_processed = df.copy()

df_processed = df_processed.drop(columns=["campaign_id"])

print("Original shape:", df.shape)
print("Shape after removing campaign_id:", df_processed.shape)

Original shape: (10000, 41)
Shape after removing campaign_id: (10000, 40)


In [7]:
df_processed["start_date"] = pd.to_datetime(
    df_processed["start_date"]
)

print(df_processed["start_date"].dtype)

datetime64[ns]


In [8]:
df_processed["start_year"] = df_processed["start_date"].dt.year
df_processed["start_month"] = df_processed["start_date"].dt.month
df_processed["start_day"] = df_processed["start_date"].dt.day

In [9]:
df_processed[
    ["start_date", "start_year", "start_month", "start_day"]
].head()

,start_date,start_year,start_month,start_day
0,2024-03-06,2024,3,6
1,2024-01-26,2024,1,26
2,2025-05-15,2025,5,15
3,2024-07-21,2024,7,21
4,2025-03-09,2025,3,9


In [10]:
df_processed = df_processed.drop(columns=["start_date"])

print("Shape:", df_processed.shape)

Shape: (10000, 42)


In [11]:
boolean_features = [
    "has_call_to_action",
    "retargeting_flag"
]

for column in boolean_features:
    df_processed[column] = df_processed[column].astype(int)

print(df_processed[boolean_features].head())
print("\nData types:")
print(df_processed[boolean_features].dtypes)

   has_call_to_action  retargeting_flag
0                   0                 0
1                   1                 0
2                   0                 0
3                   0                 0
4                   0                 0

Data types:
has_call_to_action    int64
retargeting_flag      int64
dtype: object


In [12]:
categorical_features = df_processed.select_dtypes(
    include="object"
).columns.tolist()

print("Number of categorical features:", len(categorical_features))

print("\nCategorical features:")
for column in categorical_features:
    print("-", column)

Number of categorical features: 17

Categorical features:
- campaign_objective
- platform
- ad_placement
- device_type
- operating_system
- creative_format
- creative_size
- ad_copy_length
- creative_emotion
- target_audience_age
- target_audience_gender
- audience_interest_category
- income_bracket
- purchase_intent_score
- day_of_week
- industry_vertical
- budget_tier


In [13]:
numerical_features = df_processed.select_dtypes(
    include=np.number
).columns.tolist()

print("Number of numerical features:", len(numerical_features))

print("\nNumerical features:")
for column in numerical_features:
    print("-", column)

Number of numerical features: 25

Numerical features:
- has_call_to_action
- creative_age_days
- retargeting_flag
- quarter
- hour_of_day
- campaign_day
- quality_score
- actual_cpc
- impressions
- clicks
- conversions
- ad_spend
- revenue
- bounce_rate
- avg_session_duration_seconds
- pages_per_session
- CTR
- CPC
- conversion_rate
- CPA
- ROAS
- profit
- start_year
- start_month
- start_day


In [14]:
print("Categorical features:")
for column in categorical_features:
    print("-", column)

Categorical features:
- campaign_objective
- platform
- ad_placement
- device_type
- operating_system
- creative_format
- creative_size
- ad_copy_length
- creative_emotion
- target_audience_age
- target_audience_gender
- audience_interest_category
- income_bracket
- purchase_intent_score
- day_of_week
- industry_vertical
- budget_tier


In [15]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("Preprocessing transformer created successfully.")

Preprocessing transformer created successfully.


In [17]:
# Create ROI Target

df_processed["ROI"] = (
    (df_processed["revenue"] - df_processed["ad_spend"])
    / df_processed["ad_spend"]
) * 100

print(df_processed["ROI"].describe())

count    10000.000000
mean       652.494367
std       1498.516912
min       -100.000000
25%         19.679746
50%        214.644531
75%        695.258541
max      48630.026371
Name: ROI, dtype: float64


In [18]:
# Separate Features and Targe

target = "ROI"

leakage_columns = [
    "ROI",
    "revenue",
    "profit",
    "ROAS"
]

X = df_processed.drop(columns=leakage_columns)
y = df_processed[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10000, 39)
y shape: (10000,)


In [19]:
numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()

print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical features: 22
Number of categorical features: 17

Numerical features:
['has_call_to_action', 'creative_age_days', 'retargeting_flag', 'quarter', 'hour_of_day', 'campaign_day', 'quality_score', 'actual_cpc', 'impressions', 'clicks', 'conversions', 'ad_spend', 'bounce_rate', 'avg_session_duration_seconds', 'pages_per_session', 'CTR', 'CPC', 'conversion_rate', 'CPA', 'start_year', 'start_month', 'start_day']

Categorical features:
['campaign_objective', 'platform', 'ad_placement', 'device_type', 'operating_system', 'creative_format', 'creative_size', 'ad_copy_length', 'creative_emotion', 'target_audience_age', 'target_audience_gender', 'audience_interest_category', 'income_bracket', 'purchase_intent_score', 'day_of_week', 'industry_vertical', 'budget_tier']


In [20]:
# Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (8000, 39)
X_test shape: (2000, 39)
y_train shape: (8000,)
y_test shape: (2000,)


In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)


# Fit only on training data
preprocessor.fit(X_train)

# Transform training and testing data
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train processed shape:", X_train_processed.shape)
print("X_test processed shape:", X_test_processed.shape)

X_train processed shape: (8000, 106)
X_test processed shape: (2000, 106)


In [22]:
print("Processed training data shape:", X_train_processed.shape)
print("Processed testing data shape:", X_test_processed.shape)

print("\nFirst 5 rows of processed training data:")
print(X_train_processed[:5])

print("\nData type:")
print(type(X_train_processed))

Processed training data shape: (8000, 106)
Processed testing data shape: (2000, 106)

First 5 rows of processed training data:
[[ 0.6638243   0.75505253 -0.57157662 -0.3724186   0.93784762  1.66059582
  -0.99055812 -0.70520597 -0.16501632 -0.37598758 -0.32181025 -0.48680805
   1.4049302  -1.15694432  0.25232611 -0.75129453 -0.70520597 -0.53847425
  -0.24273025 -0.9729968  -0.62159705 -1.3376274   0.          0.
   1.          0.          0.          0.          1.          0.
   0.          0.          0.          0.          0.          1.
   0.          0.          0.          0.          1.          0.
   1.          0.          0.          0.          0.          0.
   1.          0.          0.          0.          0.          0.
   0.          0.          1.          0.          0.          1.
   0.          0.          1.          0.          0.          0.
   0.          0.          0.          0.          1.          0.
   0.          0.          0.          1.          0.    

In [23]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

print("\nProcessed feature names:")
for feature in feature_names:
    print("-", feature)

Number of processed features: 106

Processed feature names:
- num__has_call_to_action
- num__creative_age_days
- num__retargeting_flag
- num__quarter
- num__hour_of_day
- num__campaign_day
- num__quality_score
- num__actual_cpc
- num__impressions
- num__clicks
- num__conversions
- num__ad_spend
- num__bounce_rate
- num__avg_session_duration_seconds
- num__pages_per_session
- num__CTR
- num__CPC
- num__conversion_rate
- num__CPA
- num__start_year
- num__start_month
- num__start_day
- cat__campaign_objective_App Installs
- cat__campaign_objective_Brand Awareness
- cat__campaign_objective_Conversions
- cat__campaign_objective_Engagement
- cat__campaign_objective_Lead Generation
- cat__platform_Facebook
- cat__platform_Google Ads
- cat__platform_Instagram
- cat__platform_LinkedIn
- cat__platform_TikTok
- cat__platform_Twitter
- cat__ad_placement_Display Network
- cat__ad_placement_Feed
- cat__ad_placement_In-Stream Video
- cat__ad_placement_Search
- cat__ad_placement_Sidebar
- cat__ad_plac

In [24]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("X_train_processed_df shape:", X_train_processed_df.shape)
print("X_test_processed_df shape:", X_test_processed_df.shape)

X_train_processed_df.head()

X_train_processed_df shape: (8000, 106)
X_test_processed_df shape: (2000, 106)


,num__has_call_to_action,num__creative_age_days,num__retargeting_flag,num__quarter,num__hour_of_day,num__campaign_day,num__quality_score,num__actual_cpc,num__impressions,num__clicks,...,cat__day_of_week_Wednesday,cat__industry_vertical_E-commerce,cat__industry_vertical_Education,cat__industry_vertical_Finance,cat__industry_vertical_Gaming,cat__industry_vertical_Healthcare,cat__industry_vertical_SaaS,cat__budget_tier_High,cat__budget_tier_Low,cat__budget_tier_Medium
9254,0.663824,0.755053,-0.571577,-0.372419,0.937848,1.660596,-0.990558,-0.705206,-0.165016,-0.375988,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1561,-1.506423,1.596134,-0.571577,-0.372419,1.081971,-1.629313,-1.634509,0.189932,1.643490,0.414458,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1670,0.663824,0.678591,-0.571577,-1.252321,0.217230,-0.290397,-0.990558,-0.957472,-0.223947,-0.271098,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
6087,0.663824,0.678591,-0.571577,0.507484,-0.359264,0.321679,-0.346607,-0.941197,-0.013327,-0.147748,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6669,0.663824,1.213824,-0.571577,1.387386,1.081971,0.589463,-0.990558,-0.619761,-0.500861,-0.485912,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [25]:
print("Missing values in X_train:", X_train_processed_df.isnull().sum().sum())

print("Missing values in X_test:", X_test_processed_df.isnull().sum().sum())

print("Infinite values in X_train:", np.isinf(X_train_processed_df).sum().sum())

print("Infinite values in X_test:", np.isinf(X_test_processed_df).sum().sum())

Missing values in X_train: 0
Missing values in X_test: 0
Infinite values in X_train: 0
Infinite values in X_test: 0


In [26]:
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nMissing values in y_train:", y_train.isnull().sum())
print("Missing values in y_test:", y_test.isnull().sum())

print("\nInfinite values in y_train:", np.isinf(y_train).sum())
print("Infinite values in y_test:", np.isinf(y_test).sum())

print("\nROI training statistics:")
print(y_train.describe())

y_train shape: (8000,)
y_test shape: (2000,)

Missing values in y_train: 0
Missing values in y_test: 0

Infinite values in y_train: 0
Infinite values in y_test: 0

ROI training statistics:
count     8000.000000
mean       645.394522
std       1409.347879
min       -100.000000
25%         18.353417
50%        215.051965
75%        691.999179
max      33779.719407
Name: ROI, dtype: float64


In [27]:
# Save Preprocessor
import joblib

In [28]:
joblib.dump(
    preprocessor,
    "../models/preprocessor.pkl"
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.
